# Cognitive Similarity — Demo Notebook

Interactive exploration of the `cognitive_similarity` library.  
Runs **entirely locally** — no GPU required.  
All tensors are synthetic (seeded random) so the notebook works without IBC stimuli or HuggingFace access.

## Section 1 — Setup

In [ ]:
import json
import tempfile
from pathlib import Path

import numpy as np

from cognitive_similarity.cache import ResponseCache
from cognitive_similarity.facade import CognitiveSimilarity
from cognitive_similarity.ica_atlas import ICANetworkAtlas
from cognitive_similarity.models import ICANetwork, ICAMode, Stimulus
from cognitive_similarity.validation import ValidationSuite

# ---------------------------------------------------------------------------
# Cache directory
# In real use: point this to your Google Drive sync folder, e.g.
#   CACHE_DIR = Path("/path/to/MyDrive/cognitive-similarity-cache")
# For this demo we use a temporary directory populated with synthetic data.
# ---------------------------------------------------------------------------
CACHE_DIR = Path(tempfile.mkdtemp(prefix="cog_sim_demo_"))
print(f"Cache directory: {CACHE_DIR}")

# ---------------------------------------------------------------------------
# Build a synthetic atlas (avoids HuggingFace access)
# ---------------------------------------------------------------------------
rng = np.random.default_rng(0)
projection_matrix = rng.standard_normal((2048, 20484)).astype(np.float32)
atlas = ICANetworkAtlas.from_projection_matrix(projection_matrix)

# ---------------------------------------------------------------------------
# Instantiate CognitiveSimilarity with the synthetic atlas
# ---------------------------------------------------------------------------
cs = CognitiveSimilarity(cache_dir=str(CACHE_DIR), _atlas=atlas)
cache = ResponseCache(str(CACHE_DIR))

# ---------------------------------------------------------------------------
# Populate the cache with synthetic collapsed responses
# ---------------------------------------------------------------------------
STIMULUS_IDS = [
    "face_01", "face_02",
    "place_01", "place_02",
    "body_01", "body_02",
    "written_character_01", "written_character_02",
    "speech_01", "speech_02",
    "non_speech_01",
    "audio_segment_01", "audio_segment_02",
    "silence_01",
    "sentence_01", "sentence_02",
    "word_list_01",
    "complex_sentence_01", "complex_sentence_02",
    "simple_sentence_01",
    "motion_video_01", "motion_video_02",
    "static_image_01",
]

# Use a dummy file path per stimulus so content_hash is deterministic.
# We write a tiny sentinel file for each stimulus so the hash is stable.
SENTINEL_DIR = CACHE_DIR / "sentinels"
SENTINEL_DIR.mkdir(parents=True, exist_ok=True)

stimuli: dict[str, Stimulus] = {}
for i, sid in enumerate(STIMULUS_IDS):
    sentinel = SENTINEL_DIR / f"{sid}.txt"
    sentinel.write_text(sid)  # unique content per stimulus
    stim = Stimulus(text_path=str(sentinel), stimulus_id=sid)
    stimuli[sid] = stim
    # Store a synthetic collapsed response
    collapsed = np.random.default_rng(i).standard_normal(20484).astype(np.float32)
    cache.put_collapsed(stim, collapsed)

print(f"Loaded {len(stimuli)} synthetic stimuli into cache.")
print(f"Atlas networks: {[n.value for n in ICANetwork]}")
print("\nSetup complete — ready to explore!")

## Section 2 — Single Comparison

Compare two stimuli and inspect the per-network `CognitiveSimilarityProfile`.

In [ ]:
stim_a = stimuli["face_01"]
stim_b = stimuli["place_01"]

result = cs.compare(stim_a, stim_b)
profile = result.profile

print(f"Comparing '{result.stimulus_a_id}' vs '{result.stimulus_b_id}'")
print(f"ICA mode: {profile.ica_mode.value}")
print(f"Whole-cortex score: {profile.whole_cortex_score:.4f}")
print()
print("Per-network scores:")
for network, ns in profile.network_scores.items():
    warning_str = f"  ⚠ {ns.warning}" if ns.warning else ""
    print(f"  {network.value:<35} score={ns.score:+.4f}  vertices={ns.vertex_count}{warning_str}")

# Highest-scoring network
best_network = max(profile.network_scores, key=lambda n: profile.network_scores[n].score)
best_score = profile.network_scores[best_network].score
print(f"\nHighest similarity: {best_network.value} ({best_score:+.4f})")

## Section 3 — Ranked Similarity

Given a query stimulus, rank the corpus by cognitive similarity per network.

In [ ]:
import pandas as pd

query = stimuli["face_01"]
corpus_ids = ["face_02", "place_01", "body_01", "speech_01", "motion_video_01"]
corpus = [stimuli[sid] for sid in corpus_ids]

ranked = cs.rank(query, corpus)

print(f"Query: '{ranked.query_id}'")
print()

# Build a DataFrame: rows = corpus stimuli, columns = networks
rows = {sid: {} for sid in corpus_ids}
for network, entries in ranked.rankings_by_network.items():
    for entry in entries:
        rows[entry.stimulus_id][network.value] = f"{entry.score:+.4f} (rank {entry.rank})"

# Add whole-cortex column
for entry in ranked.rankings_whole_cortex:
    rows[entry.stimulus_id]["whole_cortex"] = f"{entry.score:+.4f} (rank {entry.rank})"

df = pd.DataFrame.from_dict(rows, orient="index")
df.index.name = "stimulus_id"
print(df.to_string())

## Section 4 — Per-Network Bar Chart

Visualize the 5 network scores for a stimulus pair side-by-side.

In [ ]:
import matplotlib.pyplot as plt

stim_a = stimuli["face_01"]
stim_b = stimuli["face_02"]
result = cs.compare(stim_a, stim_b)
profile = result.profile

networks = list(ICANetwork)
labels = [n.value.replace("_", "\n") for n in networks]
scores = [profile.network_scores[n].score for n in networks]
colors = ["#4C72B0" if s >= 0 else "#DD8452" for s in scores]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels, scores, color=colors, edgecolor="white", linewidth=0.8)

# Annotate each bar with its score
for bar, score in zip(bars, scores):
    y_offset = 0.01 if score >= 0 else -0.03
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + y_offset,
        f"{score:+.3f}",
        ha="center", va="bottom", fontsize=9, fontweight="bold",
    )

ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_ylim(-1.1, 1.1)
ax.set_ylabel("Pearson r")
ax.set_title(
    f"Cognitive Similarity Profile\n"
    f"'{result.stimulus_a_id}' vs '{result.stimulus_b_id}'  "
    f"(whole-cortex: {profile.whole_cortex_score:+.3f})"
)
plt.tight_layout()
plt.show()

## Section 5 — Validation Suite

Run the 9 expected-ordering checks and display pass/fail results.

In [ ]:
# Build a synthetic manifest.json so ValidationSuite can look up stimuli by ID
manifest = []
for sid, stim in stimuli.items():
    # Compute the content hash the same way ResponseCache does
    import hashlib
    h = hashlib.sha256()
    if stim.text_path is not None:
        with open(stim.text_path, "rb") as f:
            for chunk in iter(lambda: f.read(65536), b""):
                h.update(chunk)
    content_hash = h.hexdigest()
    manifest.append({
        "stimulus_id": sid,
        "category": sid.split("_")[0],
        "modality": "text",
        "content_hash": content_hash,
        "tensor_dir": f"tensors/{content_hash}",
    })

manifest_path = CACHE_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f"Manifest written: {manifest_path} ({len(manifest)} entries)")

# Run the validation suite — inject the SimilarityEngine directly
# (the facade's engine is the same one used by cs.compare/cs.rank)
suite = ValidationSuite(engine=cs._engine, cache=cache, manifest_path=str(manifest_path))
report = suite.run()

print(f"\nValidation Results — {report.passed}/{report.total} checks passed\n")
print(f"{'#':<3} {'Network':<35} {'Result':<8} {'Score A':>8} {'Score B':>8}  Description")
print("-" * 110)
for i, check in enumerate(report.checks, 1):
    status = "✓ PASS" if check.passed else "✗ FAIL"
    print(
        f"{i:<3} {check.network.value:<35} {status:<8} "
        f"{check.score_a:>+8.4f} {check.score_b:>+8.4f}  {check.description}"
    )

print(f"\nSummary: {report.passed}/{report.total} checks passed")
print("Note: with synthetic random data, checks may fail — this is expected.")
print("With real IBC stimuli the suite should pass all 9 checks.")

## Section 6 — Cache Inspection

List all cached stimuli, their sizes, and confirm round-trip serialization.

In [ ]:
tensors_dir = CACHE_DIR / "tensors"

print(f"Cache directory: {CACHE_DIR}")
print(f"Tensors directory: {tensors_dir}")
print()

# List all cached stimuli
rows = []
for hash_dir in sorted(tensors_dir.iterdir()):
    if not hash_dir.is_dir():
        continue
    collapsed_file = hash_dir / "collapsed.npy"
    if collapsed_file.exists():
        size_bytes = collapsed_file.stat().st_size
        arr = np.load(collapsed_file)
        rows.append({
            "hash (first 12)": hash_dir.name[:12],
            "shape": str(arr.shape),
            "dtype": str(arr.dtype),
            "size (KB)": f"{size_bytes / 1024:.1f}",
        })

df_cache = pd.DataFrame(rows)
print(f"Cached collapsed responses: {len(df_cache)}")
print(df_cache.to_string(index=False))

# ---------------------------------------------------------------------------
# Round-trip serialization demo
# ---------------------------------------------------------------------------
print("\n--- Round-trip serialization check ---")
test_stim = stimuli["face_01"]
original = cache.get_collapsed(test_stim)

# Overwrite with a known vector, reload, compare
test_vector = np.random.default_rng(999).standard_normal(20484).astype(np.float32)
cache.put_collapsed(test_stim, test_vector)
reloaded = cache.get_collapsed(test_stim)

match = np.array_equal(test_vector, reloaded)
print(f"put_collapsed → get_collapsed round-trip: {'✓ PASS (exact match)' if match else '✗ FAIL'}")
print(f"  Original dtype: {test_vector.dtype}, reloaded dtype: {reloaded.dtype}")
print(f"  Max absolute difference: {np.abs(test_vector - reloaded).max():.2e}")

# Restore original
cache.put_collapsed(test_stim, original)
print("\nOriginal response restored.")